In [1]:
import pandas as pd

In [2]:
junction = pd.read_csv("movie_lookup.csv")
director = pd.read_csv("director_extract.csv")

director_data = junction[['movieid', 'director']].dropna().copy()

director_data['director'] = director_data['director'].str.split(',')
print(director_data)

       movieid                     director
0            1                   [Don Hahn]
1            2             [Hayao Miyazaki]
2            3                [Jay Russell]
3            4                 [Steve Carr]
4            5       [Francis Ford Coppola]
...        ...                          ...
11359    11360          [Denzel Washington]
11360    11361               [Andrew Erwin]
11361    11362                  [Joel Coen]
11362    11363  [Apichatpong Weerasethakul]
11363    11364              [Clint Bentley]

[11350 rows x 2 columns]


In [5]:
director_data = director_data.explode('director')
print(director_data)

       movieid                   director
0            1                   Don Hahn
1            2             Hayao Miyazaki
2            3                Jay Russell
3            4                 Steve Carr
4            5       Francis Ford Coppola
...        ...                        ...
11359    11360          Denzel Washington
11360    11361               Andrew Erwin
11361    11362                  Joel Coen
11362    11363  Apichatpong Weerasethakul
11363    11364              Clint Bentley

[11352 rows x 2 columns]


In [7]:
print(director.columns)

Index(['directorid', 'directorname'], dtype='str')


In [9]:
movie_director = director_data.merge(
    director,
    left_on= 'director',
    right_on= 'directorname',
    how='left'
)

In [11]:
# checking the results
movie_director.head(20)

,movieid,director,directorid,directorname
0,1,Don Hahn,2317.0,Don Hahn
1,2,Hayao Miyazaki,2949.0,Hayao Miyazaki
2,3,Jay Russell,3368.0,Jay Russell
3,4,Steve Carr,745.0,Steve Carr
4,5,Francis Ford Coppola,714.0,Francis Ford Coppola
5,6,Kris Isacsson,2083.0,Kris Isacsson
6,7,Rodrigo GarcÃ­a,NaN,NaN
7,8,Stephan Elliott,2142.0,Stephan Elliott
8,9,Stewart Raffill,2714.0,Stewart Raffill
9,10,Andrew Bergman,3134.0,Andrew Bergman


In [ ]:
# director 6 broken, check which side
print(director_data[director_data['director'].str.contains('GarcÃ', na=False)])
print(director[director['directorname'].str.contains('Garc', na=False)])

       movieid                  director
6            7           Rodrigo GarcÃ­a
2471      2472           Rodrigo GarcÃ­a
3926      3927           Rodrigo GarcÃ­a
4540      4541           Rodrigo GarcÃ­a
5368      5369           Rodrigo GarcÃ­a
6555      6556  AdriÃ¡n GarcÃ­a Bogliano
7180      7181  AdriÃ¡n GarcÃ­a Bogliano
8082      8083           Rodrigo GarcÃ­a
10995    10996           Rodrigo GarcÃ­a
      directorid            directorname
469          470          Rodrigo García
2293        2294             Raul Garcia
2524        2525           Nicole Garcia
2833        2834  Adrián García Bogliano
5363        5364           Liz W. Garcia
5382        5383             Andy Garcia


In [ ]:
# fix broken values
director_data['director'] = director_data['director'].str.encode('latin1', errors='ignore').str.decode('utf-8', errors='ignore')

In [14]:
# re-running the merge
movie_director = director_data.merge(
    director,
    left_on='director',
    right_on='directorname',
    how='left'
)

In [15]:
# check whether it worked
movie_director.head(20)

,movieid,director,directorid,directorname
0,1,Don Hahn,2317.0,Don Hahn
1,2,Hayao Miyazaki,2949.0,Hayao Miyazaki
2,3,Jay Russell,3368.0,Jay Russell
3,4,Steve Carr,745.0,Steve Carr
4,5,Francis Ford Coppola,714.0,Francis Ford Coppola
5,6,Kris Isacsson,2083.0,Kris Isacsson
6,7,Rodrigo García,470.0,Rodrigo García
7,8,Stephan Elliott,2142.0,Stephan Elliott
8,9,Stewart Raffill,2714.0,Stewart Raffill
9,10,Andrew Bergman,3134.0,Andrew Bergman


In [16]:
# null check to see if there are more issues
movie_director['directorid'].isna().sum()

np.int64(2)

In [17]:
# finding the two values
movie_director[movie_director['directorid'].isna()]

,movieid,director,directorid,directorname
3142,3144,Jr.,NaN,NaN
6358,6367,Jr.,NaN,NaN


In [18]:
# check if real director is still linked correctly
movie_director[movie_director['movieid'].isin([3144, 6367])]

,movieid,director,directorid,directorname
3141,3144,Michael Landon,5338.0,Michael Landon
3142,3144,Jr.,NaN,NaN
6357,6367,Michael Landon,5338.0,Michael Landon
6358,6367,Jr.,NaN,NaN


In [19]:
movie_director.info()
movie_director.isnull().sum().sum()

movie_director = movie_director[['movieid','directorid']]

movie_director = movie_director.drop_duplicates(
    subset=['movieid', 'directorid']
)
movie_director.to_csv("movie_director.csv",index=False)

<class 'pandas.DataFrame'>
RangeIndex: 11352 entries, 0 to 11351
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   movieid       11352 non-null  int64  
 1   director      11352 non-null  str    
 2   directorid    11350 non-null  float64
 3   directorname  11350 non-null  str    
dtypes: float64(1), int64(1), str(2)
memory usage: 652.8 KB


In [20]:
movie_director.info()

<class 'pandas.DataFrame'>
RangeIndex: 11352 entries, 0 to 11351
Data columns (total 2 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   movieid     11352 non-null  int64  
 1   directorid  11350 non-null  float64
dtypes: float64(1), int64(1)
memory usage: 177.5 KB


In [22]:
# prevent these two rows surviving in the final csv
movie_director = movie_director[['movieid', 'directorid']].dropna()
movie_director = movie_director.drop_duplicates(subset=['movieid', 'directorid'])
movie_director.to_csv("movie_director.csv", index=False)

In [23]:
# change type to int
movie_director['directorid'] = movie_director['directorid'].astype(int)
movie_director.to_csv("movie_director.csv", index=False)